# Malware Detection Using Classifier Models vs Outlier Detection Techniques
By: Aya Abdelgawad

An extentsion of a previous group project in which the members were:
- Aya Abdelgawad
- Eric Ge 
- Jay Acosta
- Vandana George

## The Goal: Compare Oulier Detection vs Classification Models
In a previous version of this project, we investigated how different classification models performed for detecting the presence of malicious code in an executable. Applying machine learning will hopefully make antiviral protection more sophisticated for users, as the world of malware is constantly evolving, requiring frequent updates to antivirus software. The results of the previous project showed that the best classifiers are random forests, decision trees, and SVMs (in that order). However, when analyzing the raw data, we noticed that frequently, the legitimate executables would cluster around some typical values for a particular feature while malicious executables would have wildly different values. This inspired a further investigation, which we will now conduct, to analyze how outlier detection methods would perform for this problem, and if they would be better (in terms of correctness and performance) than classifiers.

The dataset we will be using was supplied by Max Secure Partner for a malware detection competition. It includes metadata on thousands of Portable Executable (PE) files, a common executable format on Windows machines. The first half of the features collected come from the fields of the PE header (see [the documentation](https://docs.microsoft.com/en-us/windows/win32/debug/pe-format) for more details), and the latter half come from analysis that Max Secure Partner collected while the program was running. The final field labels the program as either legitimate or not (i.e., containing malicious code). Some examples of the metrics included are the length of the header, the flags for the file, its hash, and the min/max/average entropy of the resources it uses while running.

In [ ]:
import numpy as np
import pandas as pd
import sklearn as sk
import matplotlib.pyplot as plt

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

%matplotlib inline

### Data Cleaning
In this section, we read in and the data and make sure it's all properly formatted. This section is exactly the same as in the previous project, in order to stay consistent with how the classification models were trained.

In [ ]:
# read in data using low_memory=False so it can properly read in the md5 hashes as strings
data = pd.read_csv('Kaggle-data.csv', low_memory=False)

# display description of all columns
pd.set_option('display.max_columns', None)
data.describe(include='all')

Looking at the description of our dataframe, we immediately see two columns that have nothing to do with the data and can be dropped: 'ID' (which is jus the row id) and 'Unnamed: 57' (which was likely created due to an error while reading in the csv data). Let's go ahead and get rid of those features.

In [ ]:
data = data.drop(columns=['ID', 'Unnamed: 57'])

Next, let's check the dataframe for NaNs. Notice from the output below there is only 1 record that contains any NaNs (and only for 1 feature). Since we have hundreds of thousands of records, it will not hurt us to just remove that 1 record.

In [ ]:
# print out any features that have non-0 NaNs
col_nans = data.isna().sum()
for i in range(len(col_nans)):
    if col_nans.iloc[i] > 0:
        print(f"{data.columns[i]} has {col_nans.iloc[i]} NaNs")

# drop the row with the NaN
print("# of rows before dropping: " + str(data.shape[0]))
data = data.dropna()
print("# of rows after dropping: " + str(data.shape[0]))

Many of our attributes are represented numerically, but some of them really make more sense as categorical variables. The featuers 'Characteristics', 'DllCharacteristics', and 'LoaderFlags' are actually each a set of about 16 binary flags (and would map to 16 features each). We will handle that mapping problem later on in the feature engineering section by implementing a custom one-hot-encoding method. Here, we will properly format the 'Machine' and 'Subsystem' features, which are codes that map to a particular CPU and subsystem needed to support the PE, respectively.

We'll start by cleaning the 'Machine' feature. Since there are many possible machine codes and only 8 present in our dataset, let's first review the values we have so we only need to map those values.

In [ ]:
# print all unique values for machine code
print(data['Machine'].unique())

# there's one strange machine code value...is it legitimate?
strange_machine_code = data[data['Machine'] == '3ab1aa9785d0681434766bb0ffc4a13c']['legitimate']
print(strange_machine_code)

The above output contains all valid possible values except for the strange hex one. As suspected, it belongs to an illegitimate executable. We will keep this in mind when mapping the values and be sure to label unexpected/invalid values as errors.

In [ ]:
# Mapping values in Machine to their meaning - Machine will now be categorical
machine_keys = {
  "332": "IMAGE_FILE_MACHINE_I386",
  "422": "IMAGE_FILE_MACHINE_SH4",
  "450": "IMAGE_FILE_MACHINE_THUMB",
  "452": "IMAGE_FILE_MACHINE_ARMNT",
  "512": "IMAGE_FILE_MACHINE_IA64",
  "34404": "IMAGE_FILE_MACHINE_AMD64",
  "43620": "IMAGE_FILE_MACHINE_ARM64"
}
data['Machine'] = data['Machine'].map(machine_keys).fillna("IMAGE_FILE_MACHINE_ERROR")
print("Unique values for Machine feature are now:")
print(data['Machine'].unique())

# Mapping values in Subsystem to their meaning - Subsystem will now be categorical
subsystem_keys = {
  0: "IMAGE_SUBSYSTEM_UNKNOWN",
  1: "IMAGE_SUBSYSTEM_NATIVE",
  2: "IMAGE_SUBSYSTEM_WINDOWS_GUI",
  3: "IMAGE_SUBSYSTEM_WINDOWS_CUI",
  5: "IMAGE_SUBSYSTEM_OS2_CUI",
  7: "IMAGE_SUBSYSTEM_POSIX_CUI",
  9: "IMAGE_SUBSYSTEM_WINDOWS_CE_GUI",
  10: "IMAGE_SUBSYSTEM_EFI_APPLICATION",
  11: "IMAGE_SUBSYSTEM_EFI_BOOT_SERVICE_DRIVER",
  12: "IMAGE_SUBSYSTEM_EFI_RUNTIME_DRIVER",
  13: "IMAGE_SUBSYSTEM_EFI_ROM",
  14: "IMAGE_SUBSYSTEM_XBOX",
  16: "IMAGE_SUBSYSTEM_WINDOWS_BOOT_APPLICATION"
}
data['Subsystem'] = data['Subsystem'].map(subsystem_keys).fillna("IMAGE_SUBSYSTEM_ERROR")
print("Unique values for Subsystem feature are now:")
print(data['Subsystem'].unique())

### Data Exploration
Time to create some visualizations to get a better understanding of our data!

Let's start by getting an overview of how our variables are correlated. This will give us an idea of which variables are giving us similar information. Below, we see a correlation matrix of all our variables, with darker backgrounds representing higher correlations. 

In [ ]:
# display color-coordinated correlation matrix
correlatable = data.select_dtypes(include='number')

correlatable.corr().style.background_gradient()

Here are some highlights we found from the expansive output:
- The feature with the highest correlation to legitimacy was 'MajorSubsystemVersion', indicating that this is likely an important feature to keep for our analysis. Intuitively, this makes sense because we expect that older systems are more vulnerable to malware. 
- For the statistics measured, they come in 3's: mean, min, and max. As a general trend, there tended to be a high correlation between mean & min and mean & max. Intuitively, this makes sense because we expect the mean to summarize our entire data. We can utilize this by only using the mean feature in our training and not have to keep min or max.
- In the optional header, all of the entries have a fixed size, except the last one: 'NumberOfRvaAndSizes'. It tells us how many directory entries are in the header. 'SizeOfOptionalHeader' gives the total size of the header. Intuitively, these two variables are telling us very similar things (how big is the whole thing vs. how big is it beyond some fixed size). However, the correlation between 'NumberOfRvaAndSizes' and 'SizeOfOptionalHeader' is -0.001512! This contradiction to our expectation warrants some investigation, which we shall conduct in the coming cells. First, let's visualize the variables.

In [ ]:
# Plot NumberOfRvaAndSizes vs. SizeOfOptionalHeader
plt.scatter(data['NumberOfRvaAndSizes'], data['SizeOfOptionalHeader'], alpha=0.3)
plt.xlabel('Number of Directory Entries in Header')
plt.ylabel('Size of Header')
plt.title('Number of Directories in Header vs. Size of Header')
plt.show()

We suspect that this odd behavior is caused by the malicious records. If we plot the same graph but only for legitimate and malicious records (as shown below), then we learn that all legitimate records have 16 directory entries and header sizes of 224-240 bytes (since directories are variable-sized, this variation is not surprising or alarming). The graph of malicious executables strongly resembles the one where we included all our dataset, indicating that the odd behavior originates from our malicious records. This trend is useful to keep in mind, as it indicates these features may be helpful for classification.

In [ ]:
# plot for legitimate executables
legit = data[data['legitimate'] == 1]
plt.scatter(legit['NumberOfRvaAndSizes'], legit['SizeOfOptionalHeader'], alpha=0.3)
plt.xlabel('Number of Directory Entries in Header')
plt.ylabel('Size of Header')
plt.title('Number of Directories in Header vs. Size of Header in LEGITIMATE Executables')
plt.show()

# plot for malicious executables
not_legit = data[data['legitimate'] == 0]
plt.scatter(not_legit['NumberOfRvaAndSizes'], not_legit['SizeOfOptionalHeader'], alpha=0.3)
plt.xlabel('Number of Directory Entries in Header')
plt.ylabel('Size of Header')
plt.title('Number of Directories in Header vs. Size of Header in MALICIOUS Executables')
plt.show()

As mentioned earlier, we believe older systems should be more vulnerable to malware. This seems to be the case for MajorSubsystemVersion, based on the correlation noted earlier and by the boxplot below. 

In [ ]:
# sanity check: intuition says older subsys more vulnerable to malware
# let's see if this is true for our data
data.boxplot(column=['MajorSubsystemVersion'], by=['legitimate'])
# yeah most of data on lower end

We want to check if this trend also appears with the operating system version. However, when we make a boxplot, there are some strange outliers in the malicious records (unsurprisingly), so let's filter out the invalid values (the max valid value is 10). After filtering out invalid values, we see that malicious records do tend to belong to older systems.

In [ ]:

data.boxplot(column=['MajorOperatingSystemVersion'], by=['legitimate'])
# have some weird outliers in non-legit data, 
# so let's look at only valid values (10 or less)
noout = data[data['MajorOperatingSystemVersion'] < 11]
noout.boxplot(column=['MajorOperatingSystemVersion'], by=['legitimate'])
# yeah most of data on lower end

Let's also confirm that this trend applies with the linker version as well. Surprisingly, this doesn't seem to apply for the linker version, indicating we may not want to use it. Note we still see the greater variation among malicious record that we have come to expect.

In [ ]:
# sanity check: intuition says older linker more vulnerable to malware
# let's see if this is true for our data
data.boxplot(column=['MajorLinkerVersion'], by=['legitimate'])
# have some weird outliers in non-legit data -> remove
noout = data[data['MajorLinkerVersion'] < 50]
noout.boxplot(column=['MajorLinkerVersion'], by=['legitimate'])
# not strictly align with intuition, but noticing that there is a lot of variation in non-legit

### Feature Selection & Engineering
In this section, we will choose which features to drop based on their relevance to the problem of trying to detect malware. Some of these features will also need to be engineered, such as binning for the versioning features and implementing a custom one-hot-encoding method for the flag features.

Because 'MajorSubsytemVersion' follows a nice clean trend and doesn't have major outliers, we think two simple bins will work fine to describe the linker version: 5 or less is "old", greater than that is "new". However, we know that 'MajorOperatingSystemVersion' has some extremely strange values and large fluctionations, so that one we will bin 5 or less as "old", 6-10 as "new", and >10 as "invalid" because those do not currently correspond to real OS versions.

In [ ]:
# bin subsys version
data['SubsystemVersion'] = pd.cut(
    data['MajorSubsystemVersion'], 
    [0, 5, max(data['MajorSubsystemVersion'])],
    labels=['old', 'new'],
    include_lowest=True)

# bin OS version
data['OSVersion'] = pd.cut(
    data['MajorOperatingSystemVersion'], 
    [0, 5, 10, max(data['MajorOperatingSystemVersion'])],
    labels=['old', 'new', 'invalid'],
    include_lowest=True)

# make sure no NaNs introduced
print ("Number of NaNs in SubsystemVersion:", data['SubsystemVersion'].isna().sum ())
print ("Number of NaNs in OSVersion:", data['OSVersion'].isna().sum ())

Although 'Characteristics', 'DllCharacteristics', and 'LoaderFlags' are all flag features, we will only be implementing the custom one-hot-encoding for 'DllCharacteristics' because we do not want to keep all of the new features that would be created by feature engineering all three. This would increase the number of dimensions we have beyond a reasonable point. We choose to keep 'DllCharacterists' rather than any of the other flag features because we are interested in having context about the linker (because there are attacks that exploit linkers), and 'MajorLinkerVersion' did not seem as useful to us.

In [ ]:
dll_flag_offsets = {
    5: "IMAGE_DLL_CHARACTERISTICS_HIGH_ENTROPY_VA", 
    6: "IMAGE_DLLCHARACTERISTICS_DYNAMIC_BASE",
    7: "IMAGE_DLLCHARACTERISTICS_FORCE_INTEGRITY",
    8: "IMAGE_DLLCHARACTERISTICS_NX_COMPAT",
    9: "IMAGE_DLLCHARACTERISTICS_NO_ISOLATION",
    10: "IMAGE_DLLCHARACTERISTICS_NO_SEH",
    11: "IMAGE_DLLCHARACTERISTICS_NO_BIND",
    12: "IMAGE_DLL_CHARACTERISTICS_APPCONTAINER",
    13: "IMAGE_DLLCHARACTERISTICS_WDM_DRIVER",
    14: "IMAGE_DLL_CHARACTERISTICS_GUARD_CF",
    15: "IMAGE_DLLCHARACTERISTICS_TERMINAL_SERVER_AWARE"}

for flag_offset in dll_flag_offsets:
    data[dll_flag_offsets[flag_offset]] = data['DllCharacteristics'].apply(lambda f: f >> flag_offset & 1)

We also need to do the normal one-hot-encode for our categorical varaibles: 'Machine' and 'Subsystem'.

In [ ]:
encoded_data = pd.get_dummies(data, columns=['Subsystem', 'Machine'], drop_first=True)

Based on our analysis and understanding of the features, we think the features below will not be as helpful for determining if code is malicious (or, in the case of 'md5', it was just too much of a hassle to try to create such a large dataframe with every hash one-hot-encoded).

In [ ]:
# will get rid of the following for preliminary cleaning:
clean_out = ['md5', 'Characteristics', 'MajorLinkerVersion', 'MinorLinkerVersion', 'SectionAlignment', 'FileAlignment', 'MajorOperatingSystemVersion', 'MinorOperatingSystemVersion', 'MajorImageVersion', 'MinorImageVersion', 'MajorSubsystemVersion', 'MinorSubsystemVersion', 'CheckSum', 'DllCharacteristics', 'SizeOfStackReserve', 'SizeOfStackCommit', 'SizeOfHeapReserve', 'SizeOfHeapCommit', 'LoaderFlags', 'SectionsMinEntropy', 'SectionsMaxEntropy', 'SectionsMinRawsize', 'SectionMaxRawsize', 'SectionsMinVirtualsize', 'SectionMaxVirtualsize', 'ResourcesMinEntropy', 'ResourcesMaxEntropy', 'ResourcesMinSize', 'ResourcesMaxSize']

# print out the number of features dropped
print("Features dropped:", len(clean_out))

data = encoded_data.drop(columns=clean_out)

# how many features do we have now?
print("Number of features in dataframe:", data.shape[1])

# check head to make sure features are as expect
data.head()


### Trained Classifiers
We will now measure the accuracy and performance of the best-performing classifiers that we discovered through our previous project. The classifiers we tested were:
- Decision Trees
- Naive Bayes
- K-Nearest Neighbors
- Neural Networks
- Random Forest
- SVM

However, before we jump into the classification portion, we need to address a few issues: class imbalance, and the scoring metric.

##### Mitigating Class Imbalance

Our dataset has a slight class imbalance. To mitigate this, we chose to undersample our dataset and run our models against this undersampled data.

In [ ]:
num_legit = 75502
deep_copy_data = data.copy()
random_pruned_data = deep_copy_data.groupby("legitimate").sample(n=num_legit, random_state=2)
labels = random_pruned_data["legitimate"]
features = random_pruned_data.drop(columns = ["legitimate", 'OSVersion'])

num_legit = (random_pruned_data['legitimate'] == 1).sum()
num_malicious = (random_pruned_data['legitimate'] == 0).sum()

print ("Percent Malicious:", num_malicious / len (random_pruned_data))
print ("Percent Safe:", num_legit / len (random_pruned_data))
random_pruned_data.head ()

##### Helper Methods

Next, we will define to some utility methods for evaluating our models. The `print_confusion_matrix` method will display a stylized table of a proximity matrix. `split_feats_labels` will split data into labels and features.  Lastly, the `get_scorer` method allows us to apply a cost matrix to a confusion matrix and get the cost of a model.


Additionally, we have also included the helper method `get_k_feat_names`, which returns the names of features from a `SelectKBest` object, which is sklearn's method of picking the best _K_ features to run on a dataset.

In [ ]:
# from homework
def print_confusion_matrix(TN, FP, FN, TP):
    table_data = [[TP,FN],[FP,TN]]
    df = pd.DataFrame(table_data, columns =['Predicted Safe','Predicted Unsafe'])
    df = df.rename(index={0: 'Actual Safe', 1: 'Actual Unsafe'})
    display(df)

# written by Jay
def split_feats_labels (data, class_label='legitimate'):
    return data.drop (columns=[class_label]), data[class_label]

def get_k_feat_names (KBest, data):
    cols = KBest.get_support(indices=True)
    return [name for name in data.iloc[:,cols]]

from sklearn.metrics import make_scorer, confusion_matrix

"""
Cost Matrix: 
    pass in costs based on penalty for 
    these evaluations [TN, FP, FN, TP]

    when doing greater_is_better, sklearn chooses to have + as a penalty, and - as a reward
    reward correct classifications
    lighter penalty for marking malicious but actually safe
    harsher penalty for marking safe but actually malicious
"""
def get_scorer (cost_matrix=[-10, 100, 50, -10]):
    
    def calc_cost (y_true, y_pred, **kwargs):
        return (cost_matrix * confusion_matrix(y_true, y_pred).ravel()).sum ()
    
    return make_scorer (calc_cost, greater_is_better=False)

score = get_scorer ()

#### Decision Trees
A bit of housekeeping before we train our decision tree: scikitlearn's decision trees to do not like working with non-discrete bins. Rather than discretizing the bins, this is an opportunity for us to allow the decision tree to bin the values for us and see if our bins are similar.

In [59]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

random_pruned_data = deep_copy_data.groupby("legitimate").sample(n=num_legit, random_state=2)
labels = random_pruned_data["legitimate"]
features = random_pruned_data.drop(columns = ["legitimate", 'OSVersion'])

# decision tree doensn't like working with pre-binned data,
# so we will let it bin the data for us during training
random_tree_data = deep_copy_data.groupby("legitimate").sample(n=num_legit, random_state=2)
random_tree_data = random_tree_data.drop(columns=['OSVersion', 'SubsystemVersion'])
random_tree_data['OSVersion'] = encoded_data['MajorOperatingSystemVersion']
random_tree_data['SubsystemVersion'] = encoded_data['MajorSubsystemVersion']

# separate features and labels
features = random_tree_data
labels = features.pop('legitimate')

This training phase was done over several stages, tweaking the parameters to provide for the grid search. As we went, we removed paramters that consistently did not give optimal results, such as a `max depth` less than 20 or using sqrt (or log2) for `max_features`.

In [61]:
# make a decision tree and train on data
dt = DecisionTreeClassifier(criterion='gini')

params = [{'max_depth': range(20, 41, 5),
         'max_features': [0.7, 0.8, 0.9, None],
         'min_impurity_decrease': [0.001, 0.0001]}]

best_dt = GridSearchCV(dt,
                      param_grid=params,
                      scoring=score,
                      cv=5)
best_dt.fit(features,labels)
print(best_dt.best_params_)

{'max_depth': 20, 'max_features': 0.8, 'min_impurity_decrease': 0.0001}


When run over multiple iterations, we found best parameters are a max depth of around 20, using 80% of the features during a split, and setting the minimum impurity decrease to about 0.0001. This tree is too large to clearly draw, but we can print out which are the most important features (the features that were split on very high up in the tree).

In [62]:
dec_tree = DecisionTreeClassifier(max_depth=20, max_features=0.8, min_impurity_decrease=0.0001)
dec_tree.fit(features, labels)

# get evaluation metrics
scores = cross_val_score(dec_tree, features, labels, cv=10)
pred = cross_val_predict (dec_tree, features, labels, cv=10)

print ("Average accuracy:", accuracy_score (labels, pred))
print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())

print ("Classification Report:")
print (classification_report (labels, pred))

important_features = {}
for i in range(len(tree_data.columns) - 1):
    if(dec_tree.feature_importances_[i] > 0.009):
        # store the feature and its importance
        important_features[dec_tree.feature_importances_[i]] = tree_data.columns[i]

# print features in order of most important
for importance in sorted(important_features, reverse=True):
    print(important_features[importance], "importance:", importance)

Average accuracy: 0.9716894916690949


,Predicted Safe,Predicted Unsafe
Actual Safe,73487,2015
Actual Unsafe,2260,73242


Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97     75502
           1       0.97      0.97      0.97     75502

    accuracy                           0.97    151004
   macro avg       0.97      0.97      0.97    151004
weighted avg       0.97      0.97      0.97    151004

ImageBase importance: 0.7082136837603168
Subsystem_IMAGE_SUBSYSTEM_WINDOWS_GUI importance: 0.1311308428771866
OSVersion importance: 0.0409171149978919
VersionInformationSize importance: 0.014651122572742627
ResourcesMeanEntropy importance: 0.01460143938143639
ImportsNb importance: 0.014142877360273119
IMAGE_DLLCHARACTERISTICS_DYNAMIC_BASE importance: 0.010349250613395
ResourcesMeanSize importance: 0.010342388135439001
AddressOfEntryPoint importance: 0.009771787512913238


From the output above, we can see that we can get fairly high accuracy, precision, and recall from our model (best of both worlds!). We also see that the key features our model is prioritizing when making a decision include the image base (the first byte of the image to start executing), the subsystem type, and the operating system version.

#### Naive Bayes
In this section, we performed the Naive Bayes algorithm throughout our dataset. Due to the limitations of sklearn's `naive_bayes` implementations, categorical and continuous features cannot be trained within the same model. We will therefore proceed to build three Naive Bayes models. One will purely work with categorical features, another with purely continuous features, and lastly one combining the two models together using voting in ensembling techniques.

In [18]:
data.head ()

,SizeOfOptionalHeader,SizeOfCode,SizeOfInitializedData,SizeOfUninitializedData,AddressOfEntryPoint,BaseOfCode,BaseOfData,ImageBase,MajorOperatingSystemVersion,MajorSubsystemVersion,SizeOfImage,SizeOfHeaders,NumberOfRvaAndSizes,SectionsNb,SectionsMeanEntropy,SectionsMeanRawsize,SectionsMeanVirtualsize,ImportsNbDLL,ImportsNb,ImportsNbOrdinal,ExportNb,ResourcesNb,ResourcesMeanEntropy,ResourcesMeanSize,LoadConfigurationSize,VersionInformationSize,legitimate,SubsystemVersion,OSVersion,IMAGE_DLL_CHARACTERISTICS_HIGH_ENTROPY_VA,IMAGE_DLLCHARACTERISTICS_DYNAMIC_BASE,IMAGE_DLLCHARACTERISTICS_FORCE_INTEGRITY,IMAGE_DLLCHARACTERISTICS_NX_COMPAT,IMAGE_DLLCHARACTERISTICS_NO_ISOLATION,IMAGE_DLLCHARACTERISTICS_NO_SEH,IMAGE_DLLCHARACTERISTICS_NO_BIND,IMAGE_DLL_CHARACTERISTICS_APPCONTAINER,IMAGE_DLLCHARACTERISTICS_WDM_DRIVER,IMAGE_DLL_CHARACTERISTICS_GUARD_CF,IMAGE_DLLCHARACTERISTICS_TERMINAL_SERVER_AWARE,Subsystem_IMAGE_SUBSYSTEM_UNKNOWN,Subsystem_IMAGE_SUBSYSTEM_WINDOWS_BOOT_APPLICATION,Subsystem_IMAGE_SUBSYSTEM_WINDOWS_CE_GUI,Subsystem_IMAGE_SUBSYSTEM_WINDOWS_CUI,Subsystem_IMAGE_SUBSYSTEM_WINDOWS_GUI,Machine_IMAGE_FILE_MACHINE_ARM64,Machine_IMAGE_FILE_MACHINE_ARMNT,Machine_IMAGE_FILE_MACHINE_ERROR,Machine_IMAGE_FILE_MACHINE_I386,Machine_IMAGE_FILE_MACHINE_IA64,Machine_IMAGE_FILE_MACHINE_SH4,Machine_IMAGE_FILE_MACHINE_THUMB
0,224,16896,8192,0,16947,4096,24576,4194304.0,6,5,40960,1024,16,4,3.761598,6016.000000,6096.250000,3,44,0,31,1,3.492126,864.0,72,0,1,old,new,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
1,224,84480,25600,0,10973,4096,90112,65536.0,5,4,126976,1024,16,5,4.973822,22016.000000,21902.800000,2,102,100,2,1,3.486827,892.0,72,0,1,old,old,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
2,224,4608,3584,0,6452,4096,12288,264962048.0,6,6,24576,1024,16,4,3.329824,1792.000000,1708.000000,2,27,0,3,1,3.517270,952.0,72,0,1,new,new,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
3,224,108544,15872,0,105021,4096,114688,268435456.0,6,6,143360,1024,16,5,3.404831,24883.200000,25645.400000,12,66,0,105,2,3.270559,1032.0,72,0,1,new,new,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
4,224,513024,2048,0,520922,8192,524288,268435456.0,4,6,540672,512,16,3,2.978056,171690.666667,171265.333333,1,1,0,0,1,3.420977,954.0,0,0,1,new,old,1,1,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0


First, we begin by getting the names of the categorical features and the continuous features to make the separation of the dataset simpler. The first 5 records of the separated categorical and continuous datasets are also shown. _Note: The name of the variables with discrete are referring to categorical features / attributes_

In [20]:
discrete_features = [
    "OSVersion",
    *dll_flag_offsets.values(),
    *[name for name in random_pruned_data if name.startswith ('Machine')],
    *[name for name in random_pruned_data if name.startswith ('Subsystem')]
]

continuous_features = [
    name for name in data \
        if name not in discrete_features and name not in ["MajorOperatingSystemVersion", "legitimate"]]

display (random_pruned_data[discrete_features].head ())
display (random_pruned_data[continuous_features].head ())

,OSVersion,IMAGE_DLL_CHARACTERISTICS_HIGH_ENTROPY_VA,IMAGE_DLLCHARACTERISTICS_DYNAMIC_BASE,IMAGE_DLLCHARACTERISTICS_FORCE_INTEGRITY,IMAGE_DLLCHARACTERISTICS_NX_COMPAT,IMAGE_DLLCHARACTERISTICS_NO_ISOLATION,IMAGE_DLLCHARACTERISTICS_NO_SEH,IMAGE_DLLCHARACTERISTICS_NO_BIND,IMAGE_DLL_CHARACTERISTICS_APPCONTAINER,IMAGE_DLLCHARACTERISTICS_WDM_DRIVER,IMAGE_DLL_CHARACTERISTICS_GUARD_CF,IMAGE_DLLCHARACTERISTICS_TERMINAL_SERVER_AWARE,Machine_IMAGE_FILE_MACHINE_ARM64,Machine_IMAGE_FILE_MACHINE_ARMNT,Machine_IMAGE_FILE_MACHINE_ERROR,Machine_IMAGE_FILE_MACHINE_I386,Machine_IMAGE_FILE_MACHINE_IA64,Machine_IMAGE_FILE_MACHINE_SH4,Machine_IMAGE_FILE_MACHINE_THUMB,SubsystemVersion,Subsystem_IMAGE_SUBSYSTEM_UNKNOWN,Subsystem_IMAGE_SUBSYSTEM_WINDOWS_BOOT_APPLICATION,Subsystem_IMAGE_SUBSYSTEM_WINDOWS_CE_GUI,Subsystem_IMAGE_SUBSYSTEM_WINDOWS_CUI,Subsystem_IMAGE_SUBSYSTEM_WINDOWS_GUI
129622,old,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,old,0,0,0,0,1
159796,old,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,old,0,0,0,0,1
161588,old,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,old,0,0,0,0,1
187224,old,0,1,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,old,0,0,0,0,1
176476,old,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,old,0,0,0,0,1


,SizeOfOptionalHeader,SizeOfCode,SizeOfInitializedData,SizeOfUninitializedData,AddressOfEntryPoint,BaseOfCode,BaseOfData,ImageBase,SizeOfImage,SizeOfHeaders,NumberOfRvaAndSizes,SectionsNb,SectionsMeanEntropy,SectionsMeanRawsize,SectionsMeanVirtualsize,ImportsNbDLL,ImportsNb,ImportsNbOrdinal,ExportNb,ResourcesNb,ResourcesMeanEntropy,ResourcesMeanSize,LoadConfigurationSize,VersionInformationSize
129622,224,118784,394752,0,59901,4096,122880,4194304.0,532480,1024,16,5,5.660150,102707.200000,1.041830e+05,7,112,0,0,24,6.912127,14429.000000,72,14
159796,224,105472,67584,0,67963,4096,110592,4194304.0,180224,1024,16,4,4.395204,40704.000000,4.296575e+04,4,106,2,0,6,3.632021,1082.333333,72,16
161588,224,113664,682496,0,24735,4096,118784,4194304.0,851968,1024,16,3,4.263377,94549.333333,2.814293e+05,4,7,1,0,7,4.453368,80992.857143,0,0
187224,224,28672,445952,16896,14819,4096,32768,4194304.0,8708096,1024,16,6,4.136090,10410.666667,1.449848e+06,8,172,1,0,9,4.243395,1895.666667,0,13
176476,224,36864,16896,0,38896,4096,40960,4194304.0,77824,1024,16,8,2.315874,6272.000000,6.768750e+03,8,95,0,0,14,3.978249,549.785714,0,13


To begin, we are going to run Naive Bayes without any feature selection or engineering. In this example, we chose to display the continuous features to train an initial model. 

In [34]:


features, labels = split_feats_labels (random_pruned_data)

# select columns for only continuous features, drop others
ctransformer = ColumnTransformer ([("selector", "passthrough", continuous_features)], remainder="drop")

cont_pipe = Pipeline (steps=[
    ('prep', ctransformer),
    ('model', GaussianNB ()),
])

scores = cross_val_score(cont_pipe, features, labels, cv=10)
pred = cross_val_predict (cont_pipe, features, labels, cv=10)

print ("Average accuracy:", accuracy_score (labels, pred))
print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())

print ("Classification Report:")
print (classification_report (labels, pred))

Average accuracy: 0.5000529787290403


,Predicted Safe,Predicted Unsafe
Actual Safe,8,75494
Actual Unsafe,0,75502


Classification Report:
              precision    recall  f1-score   support

           0       0.50      1.00      0.67     75502
           1       1.00      0.00      0.00     75502

    accuracy                           0.50    151004
   macro avg       0.75      0.50      0.33    151004
weighted avg       0.75      0.50      0.33    151004



Based on these results, our Naive Bayes model tends to predict every itemset as unsafe. Although the accuracy of the model manages to reach 65%, the recall and f1-scores display that the features chosen for our model make it difficult for our model to come up with an accurate prediction, leading to many _unsafe_ predictions.

In addition, we want to be able to create a balance between the false positive and false negative predictions. In practical applications, false positives would be considered more harmful since our model would predict software as safe (0) when it is actually malicious (1). However, false positives should also be reduced since marking safe software as malicious would be less of a risk, but the overall model would not be useful since it would mark all software as unsafe.

To mitigate this, we used the scorer method to evaluate our models based on a cost matrix. 

#### Continuous Naive Bayes Model
In this first model, we will use the `GridSearchCV` and the `SelectKBest` feature selection class to find the optimal number of features and also the optimal features. Notice that we also use `f_classif`, which perform ANOVA to find the best K features.

In [34]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB, CategoricalNB

features, labels = split_feats_labels (random_pruned_data)

ctransformer = ColumnTransformer ([("selector", "passthrough", continuous_features)], remainder="drop")

cont_pipe = Pipeline (steps=[
    ('prep', ctransformer),
    ('kbest', SelectKBest (f_classif, k=2)),
    ('model', GaussianNB ()),
])

param_grid = {
    'kbest__k': list(range(2, len (continuous_features) + 1))
}

search = GridSearchCV (cont_pipe, param_grid, cv=10, scoring=score)
search.fit (features, labels)

best_k = search.best_params_['kbest__k']

print ("Best paramater search results:")
print (f"Using {best_k} out of {len (continuous_features)} available features")
print ("The features that the search chose is shown below:")
kbest_selector = search.best_estimator_.named_steps['kbest']
best_continuous_features = get_k_feat_names (kbest_selector, features[continuous_features])
for feat in best_continuous_features:
    print (">", feat)

Best paramater search results:
Using 5 out of 24 available features
The features that the search chose is shown below:
> SizeOfOptionalHeader
> SectionsNb
> SectionsMeanEntropy
> ResourcesMeanEntropy
> VersionInformationSize


Using the optimal number of features we found in the previous cell, we will then print out results and statistics for our trained model.

In [35]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report

features, labels = split_feats_labels (random_pruned_data)

ctransformer = ColumnTransformer ([("selector", "passthrough", continuous_features)], remainder="drop")

cont_pipe = Pipeline (steps=[
    ('prep', ctransformer),
    ('kbest', SelectKBest (f_classif, k=best_k)),
    ('model', GaussianNB ()),
])

scores = cross_val_score(cont_pipe, features, labels, cv=10, scoring=score)
pred = cross_val_predict (cont_pipe, features, labels, cv=10)

print ("Average score:", scores.mean ())
print ("Average accuracy:", accuracy_score (labels, pred))
print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())

print ("Classification Report:")
print (classification_report (labels, pred))

Average score: -132355.0
Average accuracy: 0.7090871764986358


,Predicted Safe,Predicted Unsafe
Actual Safe,35530,39972
Actual Unsafe,3957,71545


Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.95      0.77     75502
           1       0.90      0.47      0.62     75502

    accuracy                           0.71    151004
   macro avg       0.77      0.71      0.69    151004
weighted avg       0.77      0.71      0.69    151004



In this model, we have a better model than our initial model selecting all features. However, the recall and f1-score is still fairly poor. Interestingly, only about a quarter of features were selected out of all continuous features.

#### Discrete Naive Bayes Model
The next two cells repeat the experiment for discrete features. Handling data for discrete features is a little more difficult since the unique values for each category may be imbalanced. In addition, some of the labels for categories are `str` and others are `int`. To have a consistent datatype across all features, we passed the dataset into an `OrdinalEncoder`, which assigns a unique number to each category for datatypes of string.

In [37]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
import warnings

warnings.filterwarnings ('ignore')

features, labels = split_feats_labels (random_pruned_data)

dtransformer = ColumnTransformer ([("selector", "passthrough", discrete_features)], remainder="drop")
dmodel = CategoricalNB ()

disc_pipe = Pipeline (steps=[
    ('prep', dtransformer),
    ('ordi', OrdinalEncoder (handle_unknown="use_encoded_value", unknown_value=999)),
    ('kbest', SelectKBest (f_classif, k=2)),
    ('model', dmodel)
])

param_grid = {
    'kbest__k': list(range(2, len (discrete_features) + 1))
}

search = GridSearchCV (disc_pipe, param_grid, cv=10, scoring=score)
search.fit (features, labels)

best_k = search.best_params_['kbest__k']

print ("Best paramater search results:")
print (f"Using {best_k} out of {len (discrete_features)} available features")
print ("The features that the search chose is shown below:")
kbest_selector = search.best_estimator_.named_steps['kbest']
best_discrete_features = get_k_feat_names (kbest_selector, features[discrete_features])
for feat in best_discrete_features:
    print (">", feat)

Best paramater search results:
Using 6 out of 25 available features
The features that the search chose is shown below:
> OSVersion
> IMAGE_DLLCHARACTERISTICS_TERMINAL_SERVER_AWARE
> Machine_IMAGE_FILE_MACHINE_I386
> SubsystemVersion
> Subsystem_IMAGE_SUBSYSTEM_WINDOWS_CUI
> Subsystem_IMAGE_SUBSYSTEM_WINDOWS_GUI


In [38]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report

features, labels = split_feats_labels (random_pruned_data)

dtransformer = ColumnTransformer ([("selector", "passthrough", discrete_features)], remainder="drop")

disc_pipe = Pipeline (steps=[
    ('prep', dtransformer),
    ('ordi', OrdinalEncoder (handle_unknown="use_encoded_value", unknown_value=999)),
    ('kbest', SelectKBest (f_classif, k=best_k)),
    ('model', dmodel)
])

scores = cross_val_score(cont_pipe, features, labels, cv=10, scoring=score)
pred = cross_val_predict (cont_pipe, features, labels, cv=10)

print ("Average score:", scores.mean ())
print ("Average accuracy:", accuracy_score (labels, pred))

print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())

print ("Classification Report:")
print (classification_report (labels, pred))

Average score: -132355.0
Average accuracy: 0.7090871764986358


,Predicted Safe,Predicted Unsafe
Actual Safe,35530,39972
Actual Unsafe,3957,71545


Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.95      0.77     75502
           1       0.90      0.47      0.62     75502

    accuracy                           0.71    151004
   macro avg       0.77      0.71      0.69    151004
weighted avg       0.77      0.71      0.69    151004



As is shown here, a similar performance to the continuous model is seen. Interestingly, the best features that were selected related to the sizes of various parameters. This may be because of inconsistencies between sizes and other sections of execuatables.

#### Combining the Models
Now that we have results for both models, we now need to see if we can combine both models. In this section, we use a `VotingClassifier` to ensemble the results from both models. Now, we will use the dataset to make predictions using the optimal number of continuous features and another using the optimal number of categorical features. In the `VotingClassifier`, we will take a hard vote, meaning that the model employs a majority rule based on each models classifications.

In [41]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report

features, labels = split_feats_labels (random_pruned_data)

ctransformer = ColumnTransformer ([("selector", "passthrough", best_continuous_features)], remainder="drop")
dtransformer = ColumnTransformer ([("selector", "passthrough", best_discrete_features)], remainder="drop")

cmodel = GaussianNB ()
dmodel = CategoricalNB ()

cont_pipe = Pipeline (steps=[
    ('prep', ctransformer),
    ('model', cmodel)
])
disc_pipe = Pipeline (steps=[
    ('prep', dtransformer),
    ('ordi', OrdinalEncoder (handle_unknown="nan")),
    ('model', dmodel)
])

ensemble = VotingClassifier (estimators=[('continuous', cont_pipe), ('categorical', disc_pipe)], voting='hard')

scores = cross_val_score (ensemble, features, labels, cv=10, scoring=score)
pred = cross_val_predict (ensemble, features, labels, cv=10)

print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())

print ("Average score:", scores.mean ())
print ("Average accuracy:", accuracy_score (labels, pred))
print ("Classification Report:")
print (classification_report (labels, pred))

,Predicted Safe,Predicted Unsafe
Actual Safe,69846,5656
Actual Unsafe,6866,68636


Average score: 41542.0
Average accuracy: 0.9170750443696856
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.91      0.92     75502
           1       0.91      0.93      0.92     75502

    accuracy                           0.92    151004
   macro avg       0.92      0.92      0.92    151004
weighted avg       0.92      0.92      0.92    151004



The combination of the results appears to yield a higher accuracy overall, as well as a strong increase in recall. This increase in recall provides a better generalization than the initial models created using all features.

Lastly, we will perform a grid search across the entire ensemble to see if the optimal number of paramaters changed when we combined the models.

In [44]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report

features, labels = split_feats_labels (random_pruned_data)

ctransformer = ColumnTransformer ([("selector", "passthrough", best_continuous_features)], remainder="drop")
dtransformer = ColumnTransformer ([("selector", "passthrough", best_discrete_features)], remainder="drop")

cmodel = GaussianNB ()
dmodel = CategoricalNB ()

param_grid = {
    'continuous__kbest__k': list(range(2, len (continuous_features)+1)),
    'discrete__kbest__k': list(range(2, len (discrete_features)+1))
}

cont_pipe = Pipeline (steps=[
    ('prep', ctransformer),
    ('kbest', SelectKBest (f_classif, k=2)),
    ('model', cmodel)
])
disc_pipe = Pipeline (steps=[
    ('prep', dtransformer),
    ('ordi', OrdinalEncoder (handle_unknown="nan")),
    ('kbest', SelectKBest (f_classif, k=2)),
    ('model', dmodel)
])

ensemble = VotingClassifier (estimators=[('continuous', cont_pipe), ('discrete', disc_pipe)], voting='hard')

search = GridSearchCV (ensemble, param_grid, cv=10, scoring=score)
search.fit (features, labels)

print (search.best_params_)
best_k_continuous = search.best_params_['continuous__kbest__k']
best_k_discrete = search.best_params_['discrete__kbest__k']

print ("Best paramater search results:")
print (f"Using {best_k_continuous} out of {len (continuous_features)} continuous features")
print (f"Using {best_k_discrete} out of {len (discrete_features)} discrete features")

{'continuous__kbest__k': 5, 'discrete__kbest__k': 7}
Best paramater search results:
Using 5 out of 24 continuous features
Using 7 out of 26 discrete features


In [45]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report

features, labels = split_feats_labels (random_pruned_data)

ctransformer = ColumnTransformer ([("selector", "passthrough", best_continuous_features)], remainder="drop")
dtransformer = ColumnTransformer ([("selector", "passthrough", best_discrete_features)], remainder="drop")

cmodel = GaussianNB ()
dmodel = CategoricalNB ()

cont_pipe = Pipeline (steps=[
    ('prep', ctransformer),
    ('kbest', SelectKBest (f_classif, k=best_k_continuous)),
    ('model', cmodel)
])
disc_pipe = Pipeline (steps=[
    ('prep', dtransformer),
    ('ordi', OrdinalEncoder (handle_unknown="nan")),
    ('kbest', SelectKBest (f_classif, k=best_k_discrete)),
    ('model', dmodel)
])

ensemble = VotingClassifier (estimators=[('continuous', cont_pipe), ('categorical', disc_pipe)], voting='hard')

scores = cross_val_score (ensemble, features, labels, cv=10, scoring=score)
pred = cross_val_predict (ensemble, features, labels, cv=10)

print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())

print ("Average score:", scores.mean ())
print ("Average accuracy:", accuracy_score (labels, pred))
print ("Classification Report:")
print (classification_report (labels, pred))

,Predicted Safe,Predicted Unsafe
Actual Safe,69846,5656
Actual Unsafe,6866,68636


Average score: 41542.0
Average accuracy: 0.9170750443696856
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.91      0.92     75502
           1       0.91      0.93      0.92     75502

    accuracy                           0.92    151004
   macro avg       0.92      0.92      0.92    151004
weighted avg       0.92      0.92      0.92    151004



As is present in the results, we can see that the results of the grid search over the ensemble and the combination of separated optimal features is interestingly almost identical in terms of performance. 

#### K-Nearest Neighbors


In this cell, I'm running K nearest neighbors on the malicious software data. In this, I'm only running KNN and neural nets on 2,000 randomly selected rows with 1,000 legitimate and illegitimate data to keep it at a realistic runtime. As I'm running KNN, I'm runnning PCA to see how many components to reduce my data down to. I'm also looking for the optimal K value. Finally, I'm displaying some statistics including precision and recall about the models performance.

In [23]:
from sklearn.preprocessing import StandardScaler,LabelEncoder
le = LabelEncoder()

deep_copy_data = data.copy()
random_pruned_data = deep_copy_data.groupby("legitimate").sample(n=1000, random_state=2)
pd.get_dummies(random_pruned_data, columns=['SubsystemVersion'], drop_first=True)
labels = random_pruned_data["legitimate"]
features = random_pruned_data.drop(columns = ["legitimate", 'OSVersion'])
features["SubsystemVersion"]=le.fit_transform(features["SubsystemVersion"])

In [24]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

param_grid = {
    'pca__n_components': list(range(5, 19)),
    'model__n_neighbors': list(range(1, 25))
}

scalar = StandardScaler ()
pca = PCA ()
model = KNeighborsClassifier ()
pipeline = Pipeline (steps=[('scalar', scalar), ('pca', pca), ('model', model)])

search = GridSearchCV (pipeline, param_grid, cv=5, scoring = score)
search.fit (features, labels)


print ("Best Parameters")
for k in search.best_params_:
    print (" > ", k, "=", search.best_params_[k])
print ("Score using best params:", search.best_score_)

print (cross_val_score (pipeline, features, labels, cv=10, scoring=score).mean ())

pred = cross_val_predict (pipeline, features, labels, cv=10)

print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())

print ("Average accuracy:", accuracy_score (labels, pred))
print ("Classification Report:")
print (classification_report (labels, pred))

Best Parameters
 >  model__n_neighbors = 2
 >  pca__n_components = 15
Score using best params: 1690.0
826.0


,Predicted Safe,Predicted Unsafe
Actual Safe,940,60
Actual Unsafe,74,926


Average accuracy: 0.933
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.93      0.93      1000
           1       0.93      0.94      0.93      1000

    accuracy                           0.93      2000
   macro avg       0.93      0.93      0.93      2000
weighted avg       0.93      0.93      0.93      2000



Interestingly, the precision and recall on this dataset is exteremely accurate for the 1000 records run on this model. Additionally, we were able to obtain a positive score from this model from our cost matrix. Although we had to reduce the dataset drastically due to timing concerns, we believe that the performance of this model will be able to reflect the performance of a model trainined on the whole dataset.

#### Neural Networks


In this cell, I'm running a Neural Network classifier on the malicious software data. Once again, I'm only running neural nets on 2,000 randomly selected rows with 1,000 legitimate and illegitimate data to keep it at a realistic runtime. 
I'm performing a 5 fold cross validation to get a more realistic average accuracy. Finally, I'm displaying some statistics including a confusion matrix, precision, and recall about NN's performance. 

In [25]:
# your code goes here
import warnings

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score

# only displays warning once
warnings.filterwarnings ('ignore')

param_grid = {
  "model__hidden_layer_sizes": list(range (30, 70, 10)),
  "model__activation": ['logistic', 'tanh', 'relu']
}

scalar = StandardScaler ()
model = MLPClassifier ()
pipeline = Pipeline (steps=[('scalar', scalar), ('model', model)])

search = GridSearchCV (pipeline, param_grid, scoring=score)

accuracies = cross_val_score (search, features, labels, cv=5, scoring = score)

print ("Average accuracy:", accuracies.mean ())

pred = cross_val_predict (pipeline, features, labels, cv=10)

print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())

print ("Classification Report:")
print (classification_report (labels, pred))

Average accuracy: 1996.0


,Predicted Safe,Predicted Unsafe
Actual Safe,947,53
Actual Unsafe,59,941


Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.94      0.94      1000
           1       0.94      0.95      0.94      1000

    accuracy                           0.94      2000
   macro avg       0.94      0.94      0.94      2000
weighted avg       0.94      0.94      0.94      2000



Like KNN, the neural networks model performed extremely well. Additionally, the neural network model was able to achieve another positive score. Again, if timing had not been a concern during training, we believe that the performance of this model will be able to reflect the performance of a model trainined on the whole dataset.

#### Random Forest

In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import confusion_matrix, classification_report
import pickle
import warnings
import time
warnings.simplefilter("ignore")

We will build a Random Forest Classifier (RFC).
RFC will randomly select a subset of the input features
instead of examining all available features.

---

We will start by separating the data into data_Y (class label column)
and data_X (columns of all other features).
Our total data of 216351 data points: 140849 not legititmate, 75502 legitimate.
We want to deal with class imbalances in our data so we will
will use an equal number of 'legitimate' and 'not legitimate' records.
We will use Undersampling to get 75502 legitimate and 
75502 not legitimate data points. This is a total of 151,004 data points.

In [25]:
num_legit = (data['legitimate'] == 1).sum()
num_malicious = (data['legitimate'] == 0).sum()

deep_copy_data = data.copy()
random_pruned_data = deep_copy_data.groupby("legitimate").sample(n=num_legit, random_state=2)

num_legit = (random_pruned_data['legitimate'] == 1).sum()
num_malicious = (random_pruned_data['legitimate'] == 0).sum()
print ("Percent Malicious:", num_malicious / len (random_pruned_data))
print ("Percent Safe:", num_legit / len (random_pruned_data))

random_pruned_data = pd.get_dummies(random_pruned_data, columns=['SubsystemVersion','OSVersion'], drop_first=True)

# Get 151,004 data points with balanced classes 
features = random_pruned_data.copy()
labels = features.pop('legitimate')
print(features.shape)
print(labels.shape)
print(labels.value_counts())
#print(features.head())

Percent Malicious: 0.5
Percent Safe: 0.5
(151004, 50)
(151004,)
0    75502
1    75502
Name: legitimate, dtype: int64


We will use 5-fold GridSearchCV to find the best of these parameters:

Random Forests can take a long time to train, so we wanted to use a limited number of
parameters, mostly focusing on the size of the tree, the number of the trees. 
We will use criterion = Gini, because Entropy has higher computational 
cost due to the log in the equation.

Our starting parameters are:
number of trees = 100, criterion = gini, max_depth = [10,20,30]

In [34]:
def make_RFC(features, labels, param_grid,score):
    # Create RFC object
    rfc = RandomForestClassifier()
    print("start")
    start = time.time()
    grid_search = GridSearchCV(rfc, param_grid, cv=5, scoring=score)
    grid_search.fit(features,labels)
    print(grid_search.best_params_)
    end = time.time()
    print("end time: ", end-start)
    return grid_search

We created the RFC Model. The best parameters are: {'criterion': 'gini', 'max_depth': [10,20,30], 'n_estimators': 100}
We will use a 5 fold Cross Validation to predict and evaluate our model.

In [35]:
param_grid = {
        'n_estimators': [100],
        'criterion':['gini'],
        'max_depth': [10,20,30],
}

# Create RFC Model
rfc_model_1 = make_RFC(features,labels, param_grid,score)

start
{'criterion': 'gini', 'max_depth': 30, 'n_estimators': 100}
end time:  232.25876092910767


In [30]:
pred = cross_val_predict(rfc_model_1, features, labels, cv=5)
print ("Average accuracy:", accuracy_score (labels, pred))
print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())
print ("Classification Report:")
print (classification_report (labels, pred))

Average accuracy: 0.986497046435856


,Predicted Safe,Predicted Unsafe
Actual Safe,74648,854
Actual Unsafe,1185,74317


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     75502
           1       0.98      0.99      0.99     75502

    accuracy                           0.99    151004
   macro avg       0.99      0.99      0.99    151004
weighted avg       0.99      0.99      0.99    151004



The Accuracy is very high, so we decided to test this on the entire dataset,
because we think there might be some overfitting. Note that this prediction does not use nested cv prediction because of the huge dataset size and long runtimes.

In [31]:
data_copy = data.dropna(axis=0, inplace=False).copy()
data_copy = pd.get_dummies(data_copy, columns=['SubsystemVersion', 'OSVersion'], drop_first=True)
all_X = pd.DataFrame(data_copy.copy())
all_Y = pd.DataFrame(all_X.pop('legitimate'))

loaded_model = rfc_model_1

start = time.time()
result = loaded_model.predict(all_X)
end = time.time()
print("time to predict: ", end-start)
pred = pd.DataFrame(result)
pred.rename(columns = {0:'prediction'}, inplace = True)
# Print results
print("RFC Model 1 predicting entire dataset")
print_confusion_matrix (*confusion_matrix (all_Y, pred).ravel ())
print ("Classification Report:")
print (classification_report (all_Y, pred))

time to predict:  2.366046905517578
RFC Model 1 predicting entire dataset


,Predicted Safe,Predicted Unsafe
Actual Safe,75480,22
Actual Unsafe,962,139887


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    140849
           1       0.99      1.00      0.99     75502

    accuracy                           1.00    216351
   macro avg       0.99      1.00      1.00    216351
weighted avg       1.00      1.00      1.00    216351



The Accuracy and Recall is high (99%-100%). The high Accuracy and Recall rates is suspicious, so we believe we are overfitting for the dataset given. If we had more time, we would reduce the max depth of the tree to test out max_depth = [10,15,20]. If we didn't have long runtimes to consider, we would also try to increase min_samples_leaf to [11,21,31,41,51] (Odd numbers in case of majority voting) and min_impurity_decrease to 0.1.

#### SVM

We will create an SVM classifier for the data. 

Steps:
1. Scale data with StandardScaler and use PCA for dimensionality reduction.
2. Create a pipeline containing the scaler, PCA, and SVM object.
3. Use 5-fold GridSearchCV for hyperparameter tuning (for the best PCA n-components, SVM kernel functions, etc.)
4. Use 5-fold Nested Cross Validation to predict and evaluate the SVM model.
---
Hyperparameter Tuning:

For PCA, we will try the following parameters:
1. n_components range = [30,33,36,39,42,45,48]
   
For SVC, we will try the following parameters:
1. kernel function: ['linear', 'rbf']

---

Why we used only a few hyperparameters & values for hyperparameter tuning:\
We initially tried/hoped to test multiple parameters including:
1. PCA reduction with n_components in range (30,50)
2. kernel functions linear, rbf, poly, and sigmoid
3. C parameter of [0.1,1,10,100] where a lower C value means a softer margin
4. degrees [2,3,4,5] for the poly kernel function
However, due to long runtimes, we had to reduce the number of parameters.
---

Why we decided to train a small subset of the total data:\
We will use a subset of only 10,000 records, because of
long runtimes and time constraints.\
We made to sure avoid class imbalances training
a dataset with around equal number of both class labels.

---

Here are some examples of the parameters and runtime at which we interrupted the kernel:

| Dataset Size | Parameters                                                                                            | Best Parameters Found                                                                  | cv | Cross Val  Accuracy Found | Kernel Interrupted  After  |
|--------------|-------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------|----|---------------------------|----------------------------|
| 100,000      | kernel function = linear  n_components = range(30,50)                                                 | No                                                                                     | 10 | No                        | 45 min                     |
| 50,000       | kernel function = linear, n_components = [30,35,40,45,50]                                             | No                                                                                     | 10 | No                        | 7 hrs                      |
| 10,000       | kernel function = ['linear', 'rbf', 'poly'], n_components =  [25,27,29,31,33,35,37,39,41,43,45,47,49] | No                                                                                     | 10 | No                        | 1 hr                       |
| 10,000       | kernel function = linear, n_components = [30,35,40,45,50]                                             | Yes {'pca__n_components': 45,  'svc__kernel': 'linear'} Accuracy: 0.8981               | 10 | No                        | 45 min                     |
| 3000         | kernel function = ['linear', 'rbf'], n_components: [30,35,40,45,50], C parameter: [1.0,10.0]          | Yes {'pca__n_components': 35,  'svc__C': 10.0,  'svc__kernel': 'rbf'} Accuracy: 0.9156 | 5  | No                        | 2 hrs                      |

---

We continued to experiment with multiple training dataset sizes, and parameters to select the above hyperparameters and dataset sizes.

In this process, we observed that the factors affecting the runtime the most was:
1. dataset size
2. cv parameter of cross_val_score()
3. more than one non linear kernel functions
4. more than 2 hyperparameters

Note: We used scoring='accuracy', instead of the weighted score because we wanted to be able to use Pickle to store the model, but Pickle was not able to store the model if 'scoring' parameter was set to our weighted score.

In [36]:
def make_SVC(data_X, data_Y, param_grid):
    # Create a pipeline that includes scaling, PCA, and an SVC object
    scaler = StandardScaler()
    pca = PCA()
    svc = SVC()
    pipe_line = Pipeline(steps=[('scaler', scaler), ('pca', pca), ('svc', svc)] )
    
   
    # Create GridSearchCV for the inner CV loop, cv=5
    grid_search = GridSearchCV(pipe_line, param_grid, cv=5, scoring='accuracy')
    grid_search.fit(data_X, data_Y)
    print(grid_search.best_params_)
    print(grid_search.best_score_)

    
    return grid_search
    

We will use our helper function to get data set size of 10,000 
for training our SVM classifier.
So, we will get 5000 random records that are labeled 'legitimate' and
another 5000 that are 'not legitimate'.

We selected the following values for hyperparameter tuning:

pca_n_components : [30,33,36,39,42,45,48]
kernel function: ['linear', 'rbf']


In [37]:
def get_svm_random_data(data, size):       
    data_copy = pd.DataFrame(data.copy())
    # One hot encode the categorical attributes SubsystemVersion and OSVersion
    data_copy = pd.get_dummies(data_copy, columns=['SubsystemVersion', 'OSVersion'], drop_first=True)

    # get data with around an equal number of both class labels
    grouped = data_copy.groupby(data_copy.legitimate)
    data_0 = pd.DataFrame(grouped.get_group(0))
    data_1 = pd.DataFrame(grouped.get_group(1))
 
    half_size = size//2
    # data_0
    data_0X = pd.DataFrame(data_0.copy())
    # Select randomly from this data:
    data_0X = data_0X.sample(n = half_size)
    data_0Y = pd.DataFrame(data_0X.pop('legitimate'))
   
    # data_1
    data_1X = pd.DataFrame(data_1.copy())
    # Select randomly from this data:
    data_1X = data_1X.sample(n = half_size)
    data_1Y = pd.DataFrame(data_1X.pop('legitimate'))
    
   
    x = [data_0X,data_1X]
    y = [data_0Y,data_1Y]
    x = pd.concat(x)
    y = pd.concat(y)
    return x,y

We started with one svm model to test our code and runtime:\
SVM Model 1:\
Best Parameters: {'pca__n_components': 36, 'svc__kernel': 'rbf'}\
Score: 0.9438000000000001

In [39]:
# SVM MODEL 1
# Get 10,000 random records with balanced classes
features,labels = get_svm_random_data(data,10000)
print(features.shape)
print(labels.shape)
print(labels.value_counts())

# Make SVM with 10000 records and the following parameters
param_grid = {     
        'pca__n_components': [30,33,36,39,42,45,48],
        'svc__kernel': ['linear','rbf'],
}
start = time.time()
print("start")
svm_model_1 = make_SVC(features,labels, param_grid)
pred = cross_val_predict(svm_model_1, features, labels, cv=5)
print ("Average accuracy:", accuracy_score (labels, pred))
print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())
print ("Classification Report:")
print (classification_report (labels, pred))
end = time.time()
print("end: ", end-start)
# Save Model
filename = 'svm_model_1.sav'
pickle.dump(svm_model_1, open(filename, 'wb'))

(10000, 50)
(10000, 1)
legitimate
0             5000
1             5000
dtype: int64
start
{'pca__n_components': 36, 'svc__kernel': 'rbf'}
0.9438000000000001
Average accuracy: 0.9431


,Predicted Safe,Predicted Unsafe
Actual Safe,4688,312
Actual Unsafe,257,4743


Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      5000
           1       0.95      0.94      0.94      5000

    accuracy                           0.94     10000
   macro avg       0.94      0.94      0.94     10000
weighted avg       0.94      0.94      0.94     10000

end:  393.2431969642639


Once we realized the hyperparameters and data size had reasonable runtimes,
we decided to create 1 more SVM Model, trained with 10,000 records
randomly selected. We used sampling without replacement because we have more 
than enough data, and want as much variety in features without repeats in data points.

SVM Model 2:\
Best Parameters: {'pca__n_components': 39, 'svc__kernel': 'rbf'}\
Score: 0.9423999999999999

In [41]:
# SVM MODEL 2
features,labels = get_svm_random_data(data,10000)
print(features.shape)
print(labels.shape)
print(labels.value_counts())

param_grid = {     
        'pca__n_components': [30,33,36,39,42,45,48],
        'svc__kernel': ['linear','rbf'],
}
start = time.time()
print("start")
# Create Model
svm_model_2 = make_SVC(features,labels, param_grid)
# 5 cross val prediction
pred = cross_val_predict(svm_model_2, features, labels, cv=5)
print ("Average accuracy:", accuracy_score (labels, pred))
print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())
print ("Classification Report:")
print (classification_report (labels, pred))
end = time.time()
print("end: ", end-start)
# Save Model
filename = 'svm_model_2.sav'
pickle.dump(svm_model_2, open(filename, 'wb'))

(10000, 50)
(10000, 1)
legitimate
0             5000
1             5000
dtype: int64
start
{'pca__n_components': 39, 'svc__kernel': 'rbf'}
0.9423999999999999
Average accuracy: 0.9418


,Predicted Safe,Predicted Unsafe
Actual Safe,4688,312
Actual Unsafe,270,4730


Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      5000
           1       0.95      0.94      0.94      5000

    accuracy                           0.94     10000
   macro avg       0.94      0.94      0.94     10000
weighted avg       0.94      0.94      0.94     10000

end:  381.74691820144653


The Two Models have similar scores for precision, recall, accuracy, and f1, despite having chosen a different optimal number of dimensions to reduce to, which we found interesting. We wanted to improve the Accuracy and especially Recall, because when it comes to detecting malware, we need to focus on correctly detect Malware as 'not legitimate' rather 'legitimate'. There is a lower cost in predicting non-malware as malware and a higher cost in predicting malware as non-malware. So we will use the concept of an Ensemble Classifier to do a majority vote of the two SVM Classifiers.

We created a VotingClassifier from the already fitted SVM estimators (code from https://stackoverflow.com/questions/42920148/using-sklearn-voting-ensemble-with-partial-fit). 
Here is the code we would use, if we were to create a Voting Classifier from the two SVM models we created.

In [42]:
features,labels = get_svm_random_data(data,10000)
svm1 = pickle.load(open('svm_model_1.sav', 'rb'))
svm2 = pickle.load(open('svm_model_2.sav', 'rb'))
vc = VotingClassifier(estimators=[
         ('svm1', svm1), ('svm2', svm2)], voting='hard')


from sklearn.preprocessing import LabelEncoder
svm_list = [svm1, svm2]
vc.estimators_ = svm_list
vc.le_ = LabelEncoder().fit(labels)
vc.classes_ = vc.le_.classes_
result = vc.predict(features)

pred = pd.DataFrame(result)
pred.rename(columns = {0:'prediction'}, inplace = True)
print ("Average accuracy:", accuracy_score (labels, pred))
print_confusion_matrix (*confusion_matrix (labels, pred).ravel ())
print ("Classification Report:")
print (classification_report (labels, pred))

Average accuracy: 0.9458


,Predicted Safe,Predicted Unsafe
Actual Safe,4684,316
Actual Unsafe,226,4774


Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.95      5000
           1       0.95      0.94      0.95      5000

    accuracy                           0.95     10000
   macro avg       0.95      0.95      0.95     10000
weighted avg       0.95      0.95      0.95     10000



The recall results for malware are slightly better than SVM_model_1 and SVM_model_2 (226 from VC, 257 from SVM 1, 270 from SVM 2). This is important as we want to avoid malware being labeled 'legitimate'. The accuracy is also slightly better than the individual SVM models (a 1% increase). It may be better if there are more SVM Classifiers involved in the voting, which have been trained with more random samples from the data. So if we were able to deal with longer runtimes, then we would create at least 5-10 SVM classifiers, and then create a Voting Classifier out of those 10 (another hyperparameter to test). This is so that we can create multiple models that deal with a sampling of a range of our training data.

## Conclusion and Final Remarks

We evaluated our models using various metrics such as accuracy, recall, precision, f-score, and a cost matrix to determine the best performance since using only one metric could hide some aspect of the model's actual performance. An example of a model with a possible hidden poor performance was the run of the Naive Bayes algorithm with no feature selection. Although the accuracy was okay, the recall was poor since the model would consistently predict every executable is unsafe. 

Although we were able to create many impressive models to help detect malware, our best classifier (based on precision and recall) was our random forest. The accuracy is so high, we are slightly skeptical that the model has been overfitted to the data. However, our random forest consists of 100 decision trees, which are our second-best classifier. When trained, decision trees never dipped below 95% in accuracy, so it would make sense that having a hundred of them together would give almost perfect results. We tried to achieve similar results with our SVM model by using 2 togther and got minor improvements. We suspect that if we had more time and could train an ensemble classifer of many SVM's, we could get another classifier with close-to-perfect predictions.

If we were to take the opposite approach and create a simpler model, rather than a more complex one, we know which features our models deemed the most important. Some common ones include the operating system version, subsytem type, dynamic linker flags, version information size, and the mean entropy of the executables resources. This information can be used in general to help other malware detection software focus on the most effective features.

One final note is that although we chose to treat this problem as a classification problem, our group also discussed other ways to frame the problem. Based on our data, there are several features, such as `MajorOperatingSystemVersion` and `MajorSubsystemVersion`, that have a major outliers in the malicious class, making it a good candidate for outlier detection. Similarly, we could also treat this as a clustering problem and see if clustering algorithms can successfully form clusters for both safe and malcious executables. 